# 分身1号 学習ノートブック(フェーズ3)
自分の会話データ(dataset.jsonl)で Qwen3.5-4B をLoRAファインチューニングして「分身」を作る。

## 事前準備(このノートブックを動かす前に)
1. Googleドライブの「マイドライブ」に **bunshin** というフォルダを作る
2. PCの `bunshin_data\output\anon\dataset.jsonl`(仮名化済みのデータ)をその中にアップロード
3. Colabのメニュー **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選択
4. あとは上のセルから順番に ▶ を押していくだけ

⚠ 必ず **anon フォルダの方の dataset.jsonl** を使うこと(実名入りの方は上げない!)


## 1. ライブラリのインストール(2〜3分)

In [ ]:
%%capture
!pip install unsloth


## 2. Googleドライブをつなぎ、データがあるか確認

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DATA_PATH = Path("/content/drive/MyDrive/bunshin/dataset.jsonl")
SAVE_DIR = Path("/content/drive/MyDrive/bunshin")
assert DATA_PATH.exists(), "dataset.jsonl が見つからない!マイドライブのbunshinフォルダに入れてね"
print("OK! データを発見:", DATA_PATH)
print("サイズ:", f"{DATA_PATH.stat().st_size/1024:.0f} KB")


## 3. ベースモデルを読み込む(Qwen3.5-4B、4bit圧縮版)
日本語がすでに話せる40億パラメータのモデルを借りてくる。ここに「息吹らしさ」を追加学習する。

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3.5-9B",   # 3号から9Bに増強(会話の噛み合いはモデルサイズが一番効く)
    # ↑もし「CUDA out of memory」エラーが出たら "unsloth/Qwen3.5-4B" に戻す
    max_seq_length=max_seq_length,
    load_in_4bit=True,   # 4bit圧縮でT4のメモリに収める
)

# LoRA: モデル全体は凍結し、小さな「差分アダプタ」だけを学習する
# (mini_gptで学んだ過学習を防ぐ効果もある。触れるダイヤルをわざと絞る)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=1225,
)
print("モデル準備OK")


## 4. 教科書(dataset.jsonl)を学習用の形式に変換
各対話に「あなたは息吹。相手は◯◯」というシステム指示を付ける。
dataset.jsonl は会話の流れ(文脈)を保ったマルチターン形式なので、
分身は**会話の流れを読んで**、かつ**相手によって話し方を切り替えて**返せるようになる。

In [ ]:
import json
from datasets import Dataset

PROFILE = "あなたは「息吹(いぶき)」、21歳の日本人大学生。"  # 好きに編集してOK

def to_text(messages):
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False)

rows = []
with DATA_PATH.open(encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        rel = d.get("relation", "知り合い")
        system = PROFILE + f"相手は{rel}。息吹本人として、息吹のいつもの口調・文体で返信する。"
        msgs = [{"role": "system", "content": system}] + d["messages"]
        rows.append({"text": to_text(msgs)})

dataset = Dataset.from_list(rows).shuffle(seed=1225)
split = dataset.train_test_split(test_size=0.05, seed=1225)  # 5%は検証用(過学習の見張り役)
train_ds, eval_ds = split["train"], split["test"]
print(f"学習用 {len(train_ds)}件 / 検証用 {len(eval_ds)}件")
print("--- 学習データの見た目(1件) ---")
print(train_ds[0]["text"][:600])


## 5. 学習!(T4で30〜60分)
mini_gptと同じで、損失(loss)が下がっていけば順調。
eval_loss(検証損失)が上がり始めたら過学習のサイン — あの実験と同じ見方でOK。

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # 実質バッチ8
        num_train_epochs=5,              # 5周に増量。前回は最後まで検証損失が下がり続けていた(=学習不足)。
                                         # 途中で検証損失が上がり始めたら(過学習のサイン)そこで止めてOK
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=20,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        output_dir="outputs",
        seed=1225,
        report_to="none",
    ),
)

# 「相手の発言」は予測練習に使わず、「息吹の返事」の部分だけを学習する
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

trainer.train()


## 6. まず保存!(学習結果をドライブへ)
GPUの無料枠が切れてもやり直しにならないよう、**遊ぶ前に**「息吹らしさ」の本体(LoRAアダプタ、数十MB)を保存する。

In [ ]:
model.save_pretrained(str(SAVE_DIR / "bunshin_lora"))
tokenizer.save_pretrained(str(SAVE_DIR / "bunshin_lora"))
print("保存完了:", SAVE_DIR / "bunshin_lora")
print("これで学習結果は安全。ゆっくり分身と遊べます")


## 7. 分身と話してみる!🎉
`talk()` の中身を変えて何度でも試せる。relationを変えると口調が切り替わるはず。

In [ ]:
FastLanguageModel.for_inference(model)

def talk(message, relation="友達(高校からの友達)"):
    system = PROFILE + f"相手は{relation}。息吹本人として、息吹のいつもの口調・文体で返信する。"
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": message}]
    try:
        prompt = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    # text= を明示する(Qwen3.5は画像も扱えるモデルなので、
    # 位置引数で渡すと文章を画像と勘違いしてエラーになる)
    enc = tokenizer(text=prompt, return_tensors="pt")
    input_ids = enc["input_ids"].to("cuda")
    attention_mask = enc["attention_mask"].to("cuda")
    out = model.generate(input_ids=input_ids, attention_mask=attention_mask,
                         max_new_tokens=128,
                         temperature=0.7, top_p=0.9, do_sample=True)
    reply = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
    reply = reply.split("</think>")[-1].strip()  # 思考タグが混ざったら本文だけ取り出す
    print(f"[相手({relation})] {message}")
    print(f"[分身] {reply}")
    print("-" * 40)

talk("おつー、明日ひま?")
talk("最近どう?元気しとる?", relation="家族(母)")
talk("メンタルを安定させる方法を教えて", relation="AIアシスタント")


## 7.5 連続会話モード(こっちが分身の本領)
分身は「会話の流れ」ごと学習しているので、文脈ゼロの1発質問は一番苦手な条件。
`chat("...")` で会話を続けるほど自然になる。相手モードを変えるときは `reset_chat()` してから。

In [ ]:
history = []
CHAT_RELATION = "友達(高校からの友達)"   # 話し相手モード。変えたら reset_chat() すること

def reset_chat():
    global history
    history = []
    print("会話をリセットした")

def chat(message):
    system = PROFILE + f"相手は{CHAT_RELATION}。息吹本人として、息吹のいつもの口調・文体で返信する。"
    history.append({"role": "user", "content": message})
    msgs = [{"role": "system", "content": system}] + history
    try:
        prompt = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text=prompt, return_tensors="pt")
    input_ids = enc["input_ids"].to("cuda")
    attention_mask = enc["attention_mask"].to("cuda")
    out = model.generate(input_ids=input_ids, attention_mask=attention_mask,
                         max_new_tokens=128, temperature=0.7, top_p=0.9,
                         repetition_penalty=1.1, do_sample=True)
    reply = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
    reply = reply.split("</think>")[-1].strip()
    history.append({"role": "assistant", "content": reply})
    print(f"[あなた] {message}")
    print(f"[分身] {reply}")
    print("-" * 40)

chat("おつー")
chat("明日ひま?")
chat("なんか最近ハマってることある?")


### (再開用)過去に保存したLoRAから分身を復元する
※学習をやり直さずに続きから遊ぶ用。初回は飛ばしてOK。使うときはセル1〜3を実行してからこれを実行。

In [ ]:
# from unsloth import FastLanguageModel
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=str(SAVE_DIR / "bunshin_lora"),
#     max_seq_length=1024, load_in_4bit=True)
# FastLanguageModel.for_inference(model)
# print("保存済みの分身を復元しました")


## 8. (任意)GGUF形式に変換 → 自分のPCで動かす用(15〜25分)
q4_k_m形式(約2.5GB)に変換してドライブに保存する。
これをPCにダウンロードすれば、Ollamaでオフラインの分身が動く(フェーズ4)。

In [ ]:
model.save_pretrained_gguf("bunshin_gguf", tokenizer, quantization_method="q4_k_m")

import shutil, glob
gguf_file = glob.glob("bunshin_gguf/*.gguf")[0]
dest = SAVE_DIR / "bunshin-q4_k_m.gguf"
shutil.copy(gguf_file, dest)
print("ドライブに保存完了:", dest)
print("PCにダウンロードして、Claudeに「GGUFダウンロードしたよ」と伝えればフェーズ4開始!")
